# Double Responses in Decision Tasks
#### Group 17

In [1]:
import numpy as np
import bayesflow as bf
import keras

SEED = 2026
N_TRIALS = 1000
MAX_T = 3.0
DT = 0.02
DR_WINDOW = 0.25  # 250ms double-response window, from Evans et al. (2020)

PRIORS = {
    "nu": {"shape": 5, "scale": 0.5},
    "alpha1": {"shape": 5, "scale": 0.2},
    "tau": {"scale": 0.15},
}

rng = np.random.default_rng(SEED)

INFO:jax._src.xla_bridge:Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
INFO:bayesflow:Using backend 'jax'
/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Priors

In [2]:
def prior():
    nu = rng.gamma(shape=PRIORS["nu"]["shape"], scale=PRIORS["nu"]["scale"], size=2)
    alpha1 = rng.gamma(shape=PRIORS["alpha1"]["shape"], scale=PRIORS["alpha1"]["scale"])
    tau = rng.exponential(scale=PRIORS["tau"]["scale"])
    return {"nu": nu, "alpha1": alpha1, "tau": tau}

print(prior())

{'nu': array([1.57722685, 0.8259513 ]), 'alpha1': 1.2371612415922888, 'tau': 0.1927229427713679}


## 2. Simulator

In [3]:
def context():
    return {"n": N_TRIALS}

def simulate_dataset(nu, alpha1, tau, n=N_TRIALS, max_t=MAX_T, dt=DT, dr_window=DR_WINDOW):
    total_t = max_t + dr_window
    n_steps = int(total_t / dt)
    sqrt_dt = np.sqrt(dt)

    evidence0 = np.zeros(n)
    evidence1 = np.zeros(n)
    crossed0 = np.zeros(n, dtype=bool)
    crossed1 = np.zeros(n, dtype=bool)
    cross_time0 = np.full(n, np.inf)
    cross_time1 = np.full(n, np.inf)

    for step in range(1, n_steps + 1):
        t = step * dt
        evidence0 += nu[0] * dt + rng.normal(scale=sqrt_dt, size=n)
        evidence1 += nu[1] * dt + rng.normal(scale=sqrt_dt, size=n)

        newly0 = (~crossed0) & (evidence0 >= alpha1)
        newly1 = (~crossed1) & (evidence1 >= alpha1)
        cross_time0[newly0] = t
        cross_time1[newly1] = t
        crossed0 |= newly0
        crossed1 |= newly1

        if crossed0.all() and crossed1.all():
            break

    win_time = np.minimum(cross_time0, cross_time1)
    lose_time = np.maximum(cross_time0, cross_time1)
    win_time = np.where(np.isinf(win_time), max_t, win_time)

    correct = np.where(cross_time0 < cross_time1, 0.0, 1.0)
    first_response_time = win_time + tau

    gap = lose_time - win_time
    double_flag = ((gap <= dr_window) & (~np.isinf(lose_time))).astype(float)
    second_response_time = np.where(double_flag == 1.0, gap, 0.0)

    return {
        "first_response_time": first_response_time,
        "correct": correct,
        "double_flag": double_flag,
        "second_response_time": second_response_time,
    }

p = prior()
data = simulate_dataset(p["nu"], p["alpha1"], p["tau"])
print("double response rate:", data["double_flag"].mean())

double response rate: 0.522


## 3. BayesFlow Simulator + Adapter

In [4]:
simulator = bf.make_simulator([context, prior, simulate_dataset])

adapter = (
    bf.Adapter()
    .as_set(["first_response_time", "correct", "double_flag", "second_response_time"])
    .constrain(["nu", "alpha1", "tau"], lower=0)
    .standardize(include="nu", mean=2.502, std=1.105)
    .standardize(include="alpha1", mean=1.005, std=0.449)
    .standardize(include="tau", mean=0.159, std=0.165)
    .concatenate(["nu", "alpha1", "tau"], into="inference_variables")
    .concatenate(["first_response_time", "correct", "double_flag", "second_response_time"], into="summary_variables")
)

batch = simulator.sample(5)
adapted = adapter(batch)
for k, v in adapted.items():
    print(k, v.shape)

n (5, 1)
inference_variables (5, 4)
summary_variables (5, 1000, 4)


## 4. Workflow + Training
Training from scratch here takes ~2 hours. To skip straight to using an already-trained model, go to section 5 instead.

In [ ]:
workflow = bf.BasicWorkflow(
    simulator=simulator,
    adapter=adapter,
    inference_network=bf.networks.CouplingFlow(),
    summary_network=bf.networks.DeepSet(),
    inference_variables=["nu", "alpha1", "tau"],
    inference_conditions=["n"],
    summary_variables=["first_response_time", "correct", "double_flag", "second_response_time"],
)
history = workflow.fit_online(epochs=100, num_batches_per_epoch=50, batch_size=32)
workflow.approximator.save("../../results/trained_model_v2.keras")

INFO:bayesflow:Fitting on dataset instance of OnlineDataset.
INFO:bayesflow:Building on a test batch.


Epoch 1/100
13/50 ━━━━━━━━━━━━━━━━━━━━ 1:23 2s/step - loss: 5.0703

In [ ]:
f = bf.diagnostics.plots.loss(history)
f.savefig("loss_curve.png", dpi=300, bbox_inches="tight")

## 5. Load Already-Trained Model (fast path)

In [ ]:
approximator = keras.saving.load_model("../../results/trained_model_v2.keras")
print(approximator.optimizer.get_config())

## 6. Parameter Recovery Check

In [ ]:
true_params = prior()
trial_data = simulate_dataset(true_params["nu"], true_params["alpha1"], true_params["tau"])

fake_data = {
    "n": np.array([[N_TRIALS]]),
    "nu": true_params["nu"].reshape(1, -1),
    "alpha1": np.array([[true_params["alpha1"]]]),
    "tau": np.array([[true_params["tau"]]]),
    "first_response_time": trial_data["first_response_time"].reshape(1, -1),
    "correct": trial_data["correct"].reshape(1, -1),
    "double_flag": trial_data["double_flag"].reshape(1, -1),
    "second_response_time": trial_data["second_response_time"].reshape(1, -1),
}
posterior = approximator.sample(conditions=fake_data, num_samples=500)
print("true:", true_params)
print("estimated:", {k: posterior[k].mean(axis=1) for k in ["nu", "alpha1", "tau"]})

## 7. Diagnostics (calibration, SBC) - TODO, diagnostics pair

### 7.1 Simulated test sets

In [ ]:
N_TEST = 300
N_POST = 1000

test_sims_v2 = simulator.sample(N_TEST)

# ─── B) Draw posteriors on that test data ───────────────────────────────
posterior_sims_v2 = approximator.sample(
    num_samples = N_POST,
    conditions  = test_sims_v2
)

### 7.2 Simulation-Based Calibration (SBC)

If the approximator is calibrated, the rank of the true theta among its own posterior samples should be uniformly distributed across many test datasets. A U-shaped histogram = posterior too narrow (overconfident); a hump in the middle = posterior too wide (underconfident); a skew = systematic bias.


In [ ]:
import matplotlib.pyplot as plt

f_sbc_hist = bf.diagnostics.plots.calibration_histogram(
    estimates=posterior_sims_v2,
    targets=test_sims_v2,
    variable_keys=["nu", "alpha1", "tau"],
)
f_sbc_hist.suptitle("SBC rank histograms")
f_sbc_hist.tight_layout()
plt.show()

ECDF-difference version of the same check — more sensitive, and doesn't require choosing a bin count. The empirical curve should stay inside the shaded confidence band if calibration is good. Prefer this one over the histogram if the two disagree.

Also uses the **SBC batch**.

In [ ]:
f_sbc_ecdf = bf.diagnostics.plots.calibration_ecdf(
    estimates=posterior_sims_v2,
    targets=test_sims_v2,
    variable_keys=["nu", "alpha1", "tau"],
    difference=True,
)
f_sbc_ecdf.suptitle("SBC ECDF-difference")
f_sbc_ecdf.tight_layout()
plt.show()

### 7.3 Parameter recovery

Posterior point-estimate vs. the true simulated value, per parameter. Tight clustering around the y = x diagonal = good recovery. If one parameter scatters badly while the others look fine, that parameter is poorly identified by the data.

In [ ]:
f_recovery = bf.diagnostics.plots.recovery(
    estimates=posterior_sims_v2,
    targets=test_sims_v2,
    variable_keys=["nu", "alpha1", "tau"],
)
f_recovery.suptitle("Parameter recovery")
f_recovery.tight_layout()
plt.show()

### 7.4 Posterior z-score vs. contraction

Contraction = 1 - Var(posterior)/Var(prior): ~1 means the data pulled the posterior in tight relative to the prior (informative); ~0 means the posterior barely moved from the prior. Z-score = (posterior mean - true) / posterior SD: bias in SD units. Good behaviour = points clustered bottom-right (low |z|, high contraction). Middle-left = poorly identified parameter; upper/lower-left = prior-likelihood conflict; upper/lower-right = overfit to a wrong value.

In [ ]:
f_contraction = bf.diagnostics.plots.z_score_contraction(
    estimates=posterior_sims_v2,
    targets=test_sims_v2,
    variable_keys=["nu", "alpha1", "tau"],
)
f_contraction.suptitle("Posterior z-score vs. contraction")
f_contraction.tight_layout()
plt.show()

### 8.1 Data loading and pre-processing
The dataset was based on the paper's first experiment where 4 participants each completed 10,000 trials of a lexiacal decision task.

In [ ]:
import pandas as pd

df_rdm = pd.read_csv(
    "simulated_data.csv",
    usecols=["Sub", "list_id", "Correct", "Time1", "DoubleResp", "Time2"],
    dtype={
        "Sub" : "int32",
        "list_id": "int32",
        "Correct": "bool",
        "Time1": "float32",
        "DoubleResp": "bool",
        "Time2": "float32",
    },
    low_memory=False,
)

df_rdm.head()

In [ ]:
df_rdm = df_rdm.rename(
    columns={
        "Time1": "first_response_time",
        "Correct": "correct",
        "DoubleResp": "double_flag",
        "Time2": "second_response_time",
    }
)

df_rdm.head()

In [ ]:
# smaller sample of 1000 trials per participant
df_small_rdm = (
    df_rdm
    .groupby("Sub", group_keys=False)
    .sample(n=1000, replace=True, random_state=42)
    .reset_index(drop=True)
)
df_small_rdm.head(10)

In [ ]:
# Replacing NaNs with 0.0
df_small_rdm.loc[:, "second_response_time"] = (
    df_small_rdm["second_response_time"]
    .fillna(0.0)
    .astype(np.float32)
)

print("Remaining NaNs in second_response_time:", df_small_rdm["second_response_time"].isna().sum())

In [ ]:
df_small_rdm.info()

In [ ]:
df_small_rdm["Sub"].value_counts()

### 8.2 Estimation of double response rates and model parameters

In [ ]:
# double response rate per subject
dr_rate = df_small_rdm.groupby("Sub")["double_flag"].mean()

print("Mean double response rate per subject")
print((df_small_rdm.groupby("Sub")["double_flag"].mean()).round(3))

print("Median double response rate per subject")
print((df_small_rdm.groupby("Sub")["double_flag"].median()).round(3))

In [ ]:
max_n   = 1000
grouped = df_small_rdm.groupby("Sub", sort=False)
n_lists = len(grouped)

conditions = {
    # summary_variables: each is shape (n_lists, max_n, 1), dtype float32
    "correct": np.stack(
        [g["correct"].values.reshape(max_n, 1).astype(np.float32)
         for _, g in grouped],
        axis=0
    ),
    "first_response_time": np.stack(
        [g["first_response_time"].values.reshape(max_n, 1).astype(np.float32)
         for _, g in grouped],
        axis=0
    ),
    "double_flag": np.stack(
        [g["double_flag"].values.reshape(max_n, 1).astype(np.float32)
         for _, g in grouped],
        axis=0
    ),
    "second_response_time": np.stack(
        [g["second_response_time"].values.reshape(max_n, 1).astype(np.float32)
         for _, g in grouped],
        axis=0
    ),

    # inference_conditions: number of trials per list (shape (n_lists,)), dtype int32
    "n": np.array([g.shape[0] for _, g in grouped], dtype=np.int32)
}

print({k: v.shape for k, v in conditions.items()})
# => {
#   'correct':    (n_lists, 1000, 1),
#   'first_response_time':      (n_lists, 1000, 1),
#   'double_flag': (n_lists, 1000, 1),
#   'second_response_time':      (n_lists, 1000, 1),
#   'n':          (n_lists,)
# }


In [ ]:
# drawing posterior samples for real‐data conditions
posterior_real = approximator.sample(
    num_samples = 1000,
    conditions  = conditions
)

In [ ]:
# predicted double response rate per subject

# Dictionary to store results for all subjects
subject_dr_results = {}

for Sub, grp_df in df_small_rdm.groupby("Sub", sort=False):

    n_trials = grp_df.shape[0]

    # constructing the condition dictionary
    cond = {
        "first_response_time": grp_df["first_response_time"].to_numpy().reshape(1, -1, 1).astype(np.float32),
        "correct": grp_df["correct"].astype(bool).to_numpy().reshape(1, -1, 1).astype(np.float32),
        "double_flag": grp_df["double_flag"].to_numpy().reshape(1, -1, 1).astype(np.float32),
        "second_response_time": grp_df["second_response_time"].to_numpy().reshape(1, -1, 1).astype(np.float32),
        "n": np.array([n_trials], dtype=np.int32)
    }

    # Sample posterior parameters
    post = approximator.sample(conditions=cond, num_samples=1000)
    nu_draws = post["nu"].squeeze()
    alpha1_draws = post["alpha1"].squeeze()
    tau_draws = post["tau"].squeeze()

    # Simulate datasets across posterior draws to estimate predicted DR rates
    num_sims = 500
    pp_DR_rates = np.zeros(num_sims)

    for i in range(num_sims):
        sim = simulate_dataset(
            n=n_trials,
            nu=nu_draws[i],
            alpha1=float(alpha1_draws[i]),
            tau=float(tau_draws[i])
        )
        pp_DR_rates[i] = sim["double_flag"].mean()

    # Calculate empirical vs predicted mean and median rates
    emp_dr_rate = grp_df["double_flag"].mean()
    pred_dr_rate_mean = np.mean(pp_DR_rates)
    pred_dr_rate_median = np.median(pp_DR_rates)
    pred_dr_rate_std = np.std(pp_DR_rates)
    # 95% credible interval
    q_low, q_high = np.quantile(pp_DR_rates, [0.025, 0.975])
    # IQR
    q25, q75 = np.quantile(pp_DR_rates, [0.25, 0.75])
    iqr_val = q75 - q25

    # Store summary
    subject_dr_results[Sub] = {
        "empirical_rate": emp_dr_rate,
        "predicted_mean": pred_dr_rate_mean,
        "predicted_median": pred_dr_rate_median,
        "predicted_std": pred_dr_rate_std,
        "ci_95": (q_low, q_high),
        "iqr": iqr_val,
    }

    # Print summary per subject
    print(f"\nSubject: {Sub}")
    print(f"  Empirical DR Rate:       {emp_dr_rate:.3f}")
    print(
        f"  Predicted DR Rate Mean:  {pred_dr_rate_mean:.3f} ± {pred_dr_rate_std:.3f} "
        f"(95% CI: [{q_low:.3f}, {q_high:.3f}])\n"
        f"  Predicted DR Rate Median:  {pred_dr_rate_median:.3f}"
        f"(IQR: {iqr_val:.3f})"
    )

In [ ]:
# parameter estimates per subject
grouped = df_small_rdm.groupby("Sub")

for Sub, grp_df in grouped:
    print("\nSubject:", Sub)

    cond = {
        "first_response_time": grp_df["first_response_time"].to_numpy().reshape(1, -1, 1).astype(np.float32),
        "correct": grp_df["correct"].astype(bool).to_numpy().reshape(1, -1, 1).astype(np.float32),
        "double_flag": grp_df["double_flag"].to_numpy().reshape(1, -1, 1).astype(np.float32),
        "second_response_time": grp_df["second_response_time"].to_numpy().reshape(1, -1, 1).astype(np.float32),
        "n": np.array([grp_df.shape[0]], dtype=np.int32)
    }

    post = approximator.sample(conditions=cond, num_samples=1000)

    # mean point estimates
    nu_hat     = post["nu"].mean(axis=(0,1))
    alpha1_hat = post["alpha1"].mean()
    tau_hat    = post["tau"].mean()

    # 95% credible intervals
    nu_ci = np.percentile(post["nu"], [2.5, 97.5], axis=(0, 1))
    alpha1_ci = np.percentile(post["alpha1"], [2.5, 97.5])
    tau_ci = np.percentile(post["tau"], [2.5, 97.5])

    # format nu arrays by rounding to 3 decimal places
    nu_hat_str = np.round(nu_hat, 3).tolist()
    nu_0_str = np.round(nu_ci[0], 3).tolist()
    nu_1_str = np.round(nu_ci[1], 3).tolist()

    print(f"  nu_hat     = {nu_hat_str}  (95% CI nu_0: {nu_0_str}, 95% CI nu_1: {nu_1_str})")
    print(f"  alpha1_hat = {alpha1_hat:.3f}  (95% CI: [{alpha1_ci[0]:.3f}, {alpha1_ci[1]:.3f}])")
    print(f"  tau_hat    = {tau_hat:.3f}  (95% CI: [{tau_ci[0]:.3f}, {tau_ci[1]:.3f}])")

### 8.3 Diagnostics

In [ ]:
# 1) ordered list of subject IDs
group_keys     = list(grouped.groups.keys())

# 2) selecting the “first” one
first_list_id  = group_keys[0]
print("First list_id:", first_list_id)

# 3) data
data_first = grouped.get_group(first_list_id).reset_index(drop=True)
print("Data for first participant shape:", data_first.shape)

# 4) finding the index in the list
idx = group_keys.index(first_list_id)

# 5) posterior draws for the first subject
posterior_first = {
    "nu":     posterior_real["nu"][    idx, :, :],   # → (1000, 2)
    "alpha1": posterior_real["alpha1"][idx, :, 0],   # → (1000,)
    "tau":    posterior_real["tau"][   idx, :, 0]    # → (1000,)
}

print("Posterior for first participant:")
for name, arr in posterior_first.items():
    print(f"  {name:7s} → {arr.shape}")



In [ ]:
# over all subjects
group_keys = list(grouped.groups.keys())

# Loop through all subjects
for Sub in group_keys:
    print("\nProcessing subject:", Sub)

    # data
    data_subject = grouped.get_group(Sub).reset_index(drop=True)
    print("Data shape:", data_subject.shape)

    # finding the index in the list
    idx = group_keys.index(Sub)

    # posterior draws per subject
    posterior_subject = {
        "nu":     posterior_real["nu"][idx, :, :],      # (1000, 2)
        "alpha1": posterior_real["alpha1"][idx, :, 0],  # (1000,)
        "tau":    posterior_real["tau"][idx, :, 0]      # (1000,)
    }

    # posterior shapes
    print("Posterior shapes:")
    for name, arr in posterior_subject.items():
        print(f"{name:7s} → {arr.shape}")

    # pairs posterior plot per subject
    fig = bf.diagnostics.pairs_posterior(estimates=posterior_subject)
    # Tighten whitespace around subplots
    plt.tight_layout()

    # Save as a single crisp image file
    plt.savefig("posterior_pair_plot.png", dpi=300, bbox_inches="tight")
    plt.show()


In [ ]:
# standard deviations for estimated parameters
for name, arr in posterior_subject.items():
    # if the array has more than 1 dimension (like nu), aggregate over sampling/batch axes
    if arr.ndim > 1:
        std_val = np.std(arr, axis=(0))
        # Format rounded array as a list for clean printing
        std_str = np.round(std_val, 3).tolist()
    else:
        std_val = np.std(arr)
        std_str = f"{std_val:.3f}"

    print(f"{name} std = {std_str}")


In [ ]:
# ECDF on a fixed grid
def ecdf_grid(first_response_time, correct, t_grid):
    """
    Compute weighted ECDFs for incorrect (c=0) and correct (c=1) trials.
    Returns an array of shape (2, len(t_grid)) where each row is
      P(Time1 ≤ t and class==c) over the full sample size.
    """
    corr = correct.astype(bool)
    N    = len(first_response_time)
    ecdf = np.zeros((2, len(t_grid)))

    for c in (0, 1):
        mask = corr if c == 1 else ~corr
        for i, t in enumerate(t_grid):
            ecdf[c, i] = np.sum((first_response_time <= t) & mask) / N

    return ecdf

# Posterior‐predictive ECDF plot for one subject 
def plot_group_pp_ecdf(grp_df,
                       approximator,
                       likelihood_fn,
                       n_posterior=500,
                       t_max=3.0,
                       dt=0.02):
    """
    grp_df: DataFrame for one subject, must contain
      "Time1", "Correct", "DoubleResp", "Time2"
    workflow: trained BasicWorkflow (Variation 2)
    likelihood_fn: your two‐threshold likelihood, e.g. likelihood_v2
    """
    n_trials = grp_df.shape[0]

    # A) Build the single‐subject “conditions” dict
    cond = {
        "first_response_time":      grp_df["first_response_time"].to_numpy().reshape(1, -1, 1).astype(np.float32),
        "correct":    grp_df["correct"].to_numpy().reshape(1, -1, 1).astype(np.float32),
        "double_flag": grp_df["double_flag"].to_numpy().reshape(1, -1, 1).astype(np.float32),
        "second_response_time":      grp_df["second_response_time"].to_numpy().reshape(1, -1, 1).astype(np.float32),
        "n":          np.array([n_trials], dtype=np.int32)
    }

    # B) Draw posterior samples
    post = approximator.sample(conditions=cond, num_samples=n_posterior)
    nu_draws     = post["nu"].squeeze()      # (n_post, 2)
    alpha1_draws = post["alpha1"].squeeze()  # (n_post,)
    tau_draws    = post["tau"].squeeze()     # (n_post,)

    # C) Empirical ECDF on a grid
    t_grid = np.linspace(0, t_max, 101)
    e_ecdf = ecdf_grid(
        first_response_time   = grp_df["first_response_time"].values,
        correct = grp_df["correct"].values,
        t_grid  = t_grid
    )

    # D) Posterior‐predictive ECDFs
    pp_ecdfs = np.zeros((n_posterior, 2, len(t_grid)))
    for i in range(n_posterior):
        sim = likelihood_fn(
            n       = n_trials,
            nu      = nu_draws[i],
            alpha1  = float(alpha1_draws[i]),
            tau     = float(tau_draws[i]),
            max_t   = t_max,
            dt      = dt
        )
        pp_ecdfs[i] = ecdf_grid(
            first_response_time   = sim["first_response_time"],
            correct = sim["correct"],
            t_grid  = t_grid
        )

    # E) Compute 25/50/75% quantiles
    q_low, q_med, q_high = np.quantile(pp_ecdfs,
                                       [0.25, 0.5, 0.75],
                                       axis=0)

    # F) Plot
    fig, ax = plt.subplots(figsize=(5,4))
    for c, label, col in zip([0,1], ["Incorrect","Correct"], ["C0","C1"]):
        ax.plot(t_grid,     e_ecdf[c],
                color=col, lw=2, label=f"{label} (emp)")
        ax.plot(t_grid,     q_med[c],
                color=col, lw=1, ls="--",
                label=f"{label} (pred median)")
        ax.fill_between(
            t_grid,
            q_low[c],
            q_high[c],
            color=col, alpha=0.3,
            label=f"{label} (50% pred)"
        )

    ax.set_xlabel("First‐response RT (s)")
    ax.set_ylabel("Weighted ECDF")
    ax.legend(fontsize="small", ncol=2)
    ax.set_title(f"Subject {grp_df['Sub'].iloc[0]} – Posterior‐Predictive ECDF")
    plt.tight_layout()
    return fig

In [ ]:
# Loop over all subjects and plot each ECDF
for list_id in group_keys:
    grp_df = grouped.get_group(list_id).reset_index(drop=True)
    fig = plot_group_pp_ecdf(
        grp_df,
        approximator,    # your trained Variation 2 workflow
        simulate_dataset,  # your two-threshold likelihood function
        n_posterior=500,
        t_max=3.0,
        dt=0.02
    )
    plt.show()  # display each figure before moving on
